# TRACE API — Example Usage

This notebook demonstrates two modes of the TRACE API:
- **Mode 1 (NYC):** Full EM pipeline: infer Mallows mixture parameters from aggregate match statistics, then evaluate welfare.
- **Mode 3 (Chile):** Welfare-only: run Deferred Acceptance directly on observed individual preference lists and evaluate welfare.

Both examples use the data paths and configs defined in `file_config.py` and `.env`.

In [1]:
import sys, os
sys.path.insert(0, '../src')

import pandas as pd
import numpy as np

from trace import TRACE
from types_trace import DataKey, EvaluateConfig, Metric
from file_config import (
    POLISHED_DATA_DIR, RAW_DATA_DIR, CHILEAN_DATA_DIR, EXP_OUT_FOLDER,
    MAIN_AGG_APP_STATS_FILEPATH,
    MAIN_AGG_MATCH_STATS_FILEPATH, MAIN_AGG_MATCH_STATS_FILEPATH_SHEET,
    SCHOOL_INFO_STATS_FILEPATH, SCHOOL_INFO_STATS_FILEPATH_SHEET,
    ADDTL_SCHOOL_INFO_STATS_FILEPATH, ADDTL_SCHOOL_INFO_STATS_FILEPATH_SHEET,
    NYC_CONFIG_FILEPATH,
    CHILEAN_INDV_PREF_PROVINCE_FILEPATH,
    CHILEAN_SCHOOL_CAPACITY_BY_REGION_PROVINCE_FILEPATH,
    CHILE_CONFIG_FILEPATH,
)
from constants import DISTRICT_TO_BOROUGH_MAPPING, LEARNING_RATE
from constants import *   
from data_ingestion import read_data, nyc_preprocess_data, preprocess_chilean_data
from list_length import return_nyc_list_params

_em_to_generic = {v: k for k, v in EM_COLUMN_MAP.items()}

def  to_generic(df: pd.DataFrame) -> pd.DataFrame:
    """Rename em.py internal columns to TRACE generic columns."""
    renamed = df.rename(columns=_em_to_generic)
    pct = {
        c: f"pct_top_{c[len('% Matches to Choice 1-'):]}"
        for c in df.columns if c.startswith('% Matches to Choice 1-')
    }
    return renamed.rename(columns=pct)

---
## Mode 1 — NYC: Full EM Pipeline

NYC requires four input files. The match statistics file uses a non-default sheet name and
a first-row header, so we load all four files manually before handing off to TRACE.

`nyc_preprocess_data` returns DataFrames in em.py's internal column format
(`'School DBN'`, `'Residential District'`, etc.). `to_generic()` renames them to
TRACE's generic format (`'school_id'`, `'subdivision'`, etc.) before calling `set_df()`.

In [2]:

df_raw = read_data(f"{POLISHED_DATA_DIR}/{MAIN_AGG_APP_STATS_FILEPATH}")

match_stats_raw = read_data(
    f"{RAW_DATA_DIR}/{MAIN_AGG_MATCH_STATS_FILEPATH}",
    sheet=MAIN_AGG_MATCH_STATS_FILEPATH_SHEET,
    is_first_row_header=True,
)
school_info_raw = read_data(
    f"{RAW_DATA_DIR}/{SCHOOL_INFO_STATS_FILEPATH}",
    sheet=SCHOOL_INFO_STATS_FILEPATH_SHEET,
)
addtl_school_info_raw = read_data(
    f"{RAW_DATA_DIR}/{ADDTL_SCHOOL_INFO_STATS_FILEPATH}",
    sheet=ADDTL_SCHOOL_INFO_STATS_FILEPATH_SHEET,
)

print(f"Raw app stats:    {df_raw.shape}")
print(f"Match stats:      {match_stats_raw.shape}")
print(f"School info:      {school_info_raw.shape}")
print(f"Addtl school info:{addtl_school_info_raw.shape}")

Raw app stats:    (7295, 13)
Match stats:      (34, 13)
School info:      (452, 317)
Addtl school info:(78105, 9)


In [3]:
# ── Preprocess (NYC-specific) ─────────────────────────────────────────────────
# nyc_preprocess_data handles program-level seat splitting, Ratio/Rank computation,
# match rate normalization, and utilization. Returns em.py-format DataFrames.
df_em, match_stats_em, school_em = nyc_preprocess_data(
    df_raw, match_stats_raw, school_info_raw, addtl_school_info_raw
)

# Rename to TRACE generic format for set_df validation
final_agg_df  = to_generic(df_em)
match_stats_df = to_generic(match_stats_em)
school_df     = to_generic(school_em)

print(f"final_agg_df columns:   {list(final_agg_df.columns[:6])} ...")
print(f"match_stats_df columns: {list(match_stats_df.columns)}")
print(f"school_df columns:      {list(school_df.columns[:4])} ...")

Average list length from data: 6.93
final_agg_df columns:   ['school_id', 'School Name', 'School District', 'subdivision', 'n_applicants', 'n_true_applicants'] ...
match_stats_df columns: ['subdivision', 'n_students', 'pct_top_3', 'pct_top_5', 'pct_top_10', 'pct_unmatched']
school_df columns:      ['school_id', 'capacity', 'seats_ge', 'seats_swd'] ...


In [4]:
# ── Initialise TRACE and load preprocessed data ───────────────────────────────
nyc_model = TRACE(
    priority_config_fpath=f"{POLISHED_DATA_DIR}/{NYC_CONFIG_FILEPATH}",
)

nyc_model.set_df(DataKey.FINAL_AGGREGATES, final_agg_df)
nyc_model.set_df(DataKey.MATCH_STATS,      match_stats_df)
nyc_model.set_df(DataKey.SCHOOL,           school_df)


print(f"Mode: {nyc_model.mode}")   # 1
print(TRACE.validate_priority_config(nyc_model.get_priority_config()))

Mode: 1
[]


In [5]:
# ── Fit the model ───────────────────────────────────────────────────────────────
nyc_model.fit(
    K=1,
    M=2,
    max_iter=1,
    max_opt=1,
    n_jobs=32,
    seed=40,
    lr=LEARNING_RATE,
    subdivision_to_region=DISTRICT_TO_BOROUGH_MAPPING,
    list_length_params=return_nyc_list_params(),
    outfile=f"{EXP_OUT_FOLDER}/trace_nyc_example.txt",
    verbose=False,
)

params = nyc_model.get_mallows_params()
print(f"Fitted phis: {params['global_phis']}")
print(f"Weights:     {params['global_weights']}")


Initialization:


  Districts: 32
  Total students: 70795
  Global mixture components: K=1
  Max iterations of EM Algorithm: 1
  Max iterations of nonconvex optimizer: 1
  Simulations per evaluation: M=2


 EM ITERATION 1/1 
Entering the optimization of the global mixture...

  [EM iter 1/1] Optimizing phi[1/1], starting at 0.7120
    [EM iter 1/1] | phi[1/1] eval #1/1, trying phi=0.6328
      Simulation 1/2...
 Generated 70795 student rankings across 32 districts (51 chunks)
Invoked gale_shapley_per_school_numba_wrapper
      [TIMING] simulation 1/2: 9.983s
      Simulation 2/2...
 Generated 70795 student rankings across 32 districts (51 chunks)
Invoked gale_shapley_per_school_numba_wrapper

  Priority vs Lottery Analysis:
    Matched:   21963/70795 (31.0%)
    Priority-determined: 0/21963 (0.0% of matched)
    Lottery-determined:  21963/21963 (100.0% of matched)
    Unmatched: 48832/70795 (69.0%)
    Priority tier breakdown: {}
    Seat type breakdown: {'GE': np.int64(17964), 'SWD': np.int64(3999)}
  

In [6]:
nyc_results = nyc_model.evaluate(
    metrics=[Metric.GLOBAL_TOP_P_SWEEP, Metric.RANK_STATS, Metric.TOP_P_BY_CATEGORY],
    config=EvaluateConfig(
        max_p=10,
        stratify_by=['subdivision'],
        output_dir=f"{EXP_OUT_FOLDER}/trace_nyc_welfare",
    )
)

print(f"Avg rank:    {nyc_results.rank_stats['avg_rank']:.3f}")
print(f"Pct matched: {nyc_results.rank_stats['pct_matched']:.1f}%")
nyc_results.global_sweep[['p', 'top_p_pct']].set_index('p').head(10)

Avg rank:    3.653
Pct matched: 38.8%


,top_p_pct
p,
1,11.067166
2,16.672081
3,21.300939
4,25.455188
5,29.342468
6,32.626598
7,35.279328
8,37.022389
9,38.022459


In [7]:
nyc_results.top_p_sweep_by_category

{'subdivision':       p subdivision  students  top_p_rate  avg_rank  rank_variance  rank_std  \
 2     1          11    3186.0    0.071563    4.1065         7.6970    2.7744   
 23    1          30    2635.0    0.072486    4.3448         7.0435    2.6540   
 4     1          13     791.0    0.074589    3.6727         6.9532    2.6369   
 11    1           2    2658.0    0.074868    3.6255         6.2113    2.4923   
 30    1           8    2450.0    0.076735    4.0347         7.4980    2.7383   
 ..   ..         ...       ...         ...       ...            ...       ...   
 306  10          26    2797.0    0.533071    3.8128         5.8940    2.4278   
 314  10           4     795.0    0.594969    4.2185         5.9648    2.4423   
 288  10           1     535.0    0.607477    3.6185         5.8910    2.4271   
 303  10          23    1029.0    0.619048    3.7950         6.0410    2.4578   
 313  10          32     918.0    0.775599    3.3492         4.7529    2.1801   
 
      top_p

In [8]:
nyc_results.top_p_sweep_by_category.get('subdivision', pd.DataFrame()).head(20)

,p,subdivision,students,top_p_rate,avg_rank,rank_variance,rank_std,top_p_pct
2,1,11,3186.0,0.071563,4.1065,7.6970,2.7744,7.156309
23,1,30,2635.0,0.072486,4.3448,7.0435,2.6540,7.248577
4,1,13,791.0,0.074589,3.6727,6.9532,2.6369,7.458913
11,1,2,2658.0,0.074868,3.6255,6.2113,2.4923,7.486832
30,1,8,2450.0,0.076735,4.0347,7.4980,2.7383,7.673469
16,1,24,3755.0,0.076964,3.8917,6.8147,2.6105,7.696405
3,1,12,1889.0,0.080466,4.0471,7.6774,2.7708,8.046585
17,1,25,3042.0,0.086785,3.6297,5.7602,2.4000,8.678501
22,1,3,1247.0,0.091419,3.5904,6.2638,2.5028,9.141941
20,1,28,2837.0,0.091999,3.2798,5.6001,2.3664,9.199859


---
## Mode 3 — Chile: Welfare from Observed Preferences

Mode 3 skips inference entirely — we have the actual student preference lists.
TRACE runs Deferred Acceptance on them and computes welfare metrics.

The raw Chile individual data uses `mrun` (student id), `rbd` + `program_code` (school id),
and `Provincia` (subdivision, since `is_province_level=True`). We reshape to TRACE's
long format before calling `set_df()`.

In [9]:
# ── Load raw Chile files ──────────────────────────────────────────────────────
indv_raw      = read_data(f"{CHILEAN_DATA_DIR}/{CHILEAN_INDV_PREF_PROVINCE_FILEPATH}")
school_cap_raw = read_data(f"{CHILEAN_DATA_DIR}/{CHILEAN_SCHOOL_CAPACITY_BY_REGION_PROVINCE_FILEPATH}")

print(f"Individual prefs: {indv_raw.shape}")
print(f"School capacity:  {school_cap_raw.shape}")
indv_raw.head(3)

Individual prefs: (481678, 19)
School capacity:  (2604, 13)


,mrun,Region,Provincia,lon,lat,quality_georef,rbd,program_code,lottery,preference_number,matched_first_round,female,priority_student,high_performance_student,priority_already_registered,priority_sibling,priority_parent_civil_servant,priority_ex_student,integration_program_status_existing
0,2364012,Región Metropolitana de Santiago,Santiago_Sur,-70.644597,-33.509661,1,11812,131000000133,317,1,0,1,0,0,0,0,0,0,0
1,2364012,Región Metropolitana de Santiago,Santiago_Sur,-70.644597,-33.509661,1,25798,131000000133,27,2,1,1,0,0,0,0,0,0,0
2,2364012,Región Metropolitana de Santiago,Santiago_Sur,-70.644597,-33.509661,1,9484,131000000113,20,3,0,1,0,0,0,0,0,0,0


In [10]:
# ── Transform individual preferences to TRACE long format ─────────────────────
# Required columns: student_id, school_id, preference_number
# Optional:         subdivision (for stratification), priority/attribute columns

SUBDIVISION_COL = 'Provincia'   # is_province_level=True

# Priority columns present in Chile data (binary flags)
PRIORITY_COLS = [
    'priority_student',
    'priority_sibling',
    'priority_already_registered',
    'priority_parent_civil_servant',
    'priority_ex_student',
]
STUDENT_ATTR_COLS = ['female']

keep = ['mrun', 'rbd', 'program_code', 'preference_number', SUBDIVISION_COL]
keep += [c for c in PRIORITY_COLS + STUDENT_ATTR_COLS if c in indv_raw.columns]

chile_indv = indv_raw[keep].copy()
chile_indv['student_id']  = chile_indv['mrun'].astype(str)
chile_indv['school_id']   = chile_indv['rbd'].astype(str) + '_' + chile_indv['program_code'].astype(str)
chile_indv['subdivision'] = chile_indv[SUBDIVISION_COL].astype(str)
chile_indv = chile_indv.drop(columns=['mrun', 'rbd', 'program_code', SUBDIVISION_COL])

print(f"Individual df shape: {chile_indv.shape}")
print(f"Columns: {list(chile_indv.columns)}")
chile_indv.head(3)

Individual df shape: (481678, 10)
Columns: ['preference_number', 'priority_student', 'priority_sibling', 'priority_already_registered', 'priority_parent_civil_servant', 'priority_ex_student', 'female', 'student_id', 'school_id', 'subdivision']


,preference_number,priority_student,priority_sibling,priority_already_registered,priority_parent_civil_servant,priority_ex_student,female,student_id,school_id,subdivision
0,1,0,0,0,0,0,1,2364012,11812_131000000133,Santiago_Sur
1,2,0,0,0,0,0,1,2364012,25798_131000000133,Santiago_Sur
2,3,0,0,0,0,0,1,2364012,9484_131000000113,Santiago_Sur


In [11]:
# ── Transform school capacity to TRACE format ─────────────────────────────────
# Required columns: school_id, capacity

chile_school = (
    school_cap_raw
    .groupby(['rbd', 'program_code'])['total_capacity']
    .sum()
    .reset_index()
    .assign(school_id=lambda d: d['rbd'].astype(str) + '_' + d['program_code'].astype(str))
    .rename(columns={'total_capacity': 'capacity'})
    [['school_id', 'capacity']]
)

print(f"School df shape: {chile_school.shape}")
chile_school.head(3)

School df shape: (2604, 2)


,school_id,capacity
0,1_131000000131,245
1,4_131000000133,280
2,5_131000000133,120


In [12]:
# ── Initialise TRACE and load data ────────────────────────────────────────────
chile_model = TRACE(
    priority_config_fpath=f"{CHILEAN_DATA_DIR}/{CHILE_CONFIG_FILEPATH}",
)

chile_model.set_df(DataKey.INDIVIDUAL, chile_indv)
chile_model.set_df(DataKey.SCHOOL,     chile_school)

print(f"Mode: {chile_model.mode}")   # -> 3

Mode: 3


In [13]:
# ── Run matching ──────────────────────────────────────────────────────────────
# priority_attribute_cols: binary flags from the observed data passed to DA.
# Note: these column names must match the group names in the priority config
# for school-independent priority tiers to apply correctly.
present_priority_cols = [c for c in PRIORITY_COLS if c in chile_indv.columns]
present_attr_cols     = [c for c in STUDENT_ATTR_COLS if c in chile_indv.columns]

chile_outcomes = chile_model.run_matching(
    priority_attribute_cols=present_priority_cols,
    student_attribute_cols=present_attr_cols,
    seed=40,
)

print(f"Students matched: {(chile_outcomes.matches_idx >= 0).sum()} / {len(chile_outcomes.matches_idx)}")

Invoked gale_shapley_per_school_numba_wrapper
Students matched: 123985 / 124492


In [14]:

stratify = ['subdivision'] + ([c for c in present_attr_cols if c == 'female'])

chile_results = chile_model.evaluate(
    metrics=[Metric.RANK_STATS, Metric.TOP_P_BY_CATEGORY],
    config=EvaluateConfig(
        max_p=10,
        stratify_by=stratify,
        output_dir=f"{EXP_OUT_FOLDER}/trace_chile_welfare",
    )
)

print(f"Avg rank:    {chile_results.rank_stats['avg_rank']:.3f}")
print(f"Pct matched: {chile_results.rank_stats['pct_matched']:.1f}%")

Avg rank:    1.083
Pct matched: 99.6%


In [15]:
# Top-p rates by subdivision
chile_results.top_p_sweep_by_category.get('subdivision', pd.DataFrame()).head(20)

,p,subdivision,students,top_p_rate,avg_rank,rank_variance,rank_std,top_p_pct
55,1,Tamarugal,453.0,0.810155,1.0763,0.0911,0.3018,81.015453
47,1,Santiago_Nor_Poniente,4095.0,0.816361,1.2641,0.5173,0.7192,81.636142
24,1,Huasco,976.0,0.818648,1.2763,0.4507,0.6713,81.864754
30,1,Los Andes,1215.0,0.822222,1.1893,0.2020,0.4495,82.222222
39,1,Petorca,676.0,0.844675,1.1620,0.1597,0.3997,84.467456
2,1,Arica,1988.0,0.848089,1.1642,0.1737,0.4168,84.808853
21,1,El Loa,1787.0,0.860101,1.1599,0.2047,0.4525,86.010073
57,1,Tocopilla,399.0,0.882206,1.1178,0.1042,0.3228,88.220551
48,1,Santiago_Oriente,804.0,0.883085,1.1539,0.2482,0.4982,88.308458
0,1,Antofagasta,3247.0,0.884201,1.1246,0.1307,0.3615,88.420080


In [16]:
# Top-p rates by gender (if female column present)
if 'female' in stratify:
    display(chile_results.top_p_sweep_by_category.get('female', pd.DataFrame()))

,p,female,students,top_p_rate,avg_rank,rank_variance,rank_std,top_p_pct
0,1,0,64147.0,0.927697,1.0856,0.1248,0.3532,92.769732
1,1,1,60345.0,0.934062,1.0802,0.1230,0.3507,93.406247
3,2,1,60345.0,0.982351,1.0802,0.1230,0.3507,98.235148
2,2,0,64147.0,0.983507,1.0856,0.1248,0.3532,98.350663
5,3,1,60345.0,0.992576,1.0802,0.1230,0.3507,99.257602
4,3,0,64147.0,0.992829,1.0856,0.1248,0.3532,99.282897
7,4,1,60345.0,0.994946,1.0802,0.1230,0.3507,99.494573
6,4,0,64147.0,0.994965,1.0856,0.1248,0.3532,99.496469
9,5,1,60345.0,0.995675,1.0802,0.1230,0.3507,99.567487
8,5,0,64147.0,0.995697,1.0856,0.1248,0.3532,99.569738
